# Assign EphMRA Codes to Antibodies by MOA

This notebook reads the antibody list and assigns EphMRA codes based on the Mechanism of Action field.

In [2]:
%pip install regex

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 23.0 MB/s  0:00:00

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import regex as re

In [ ]:
# Save the updated CSV with EphMRA codes
output_path = 'List of Antibodies by MOA with EphMRA.csv'
antibodies.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"Saved to: {output_path}")

In [ ]:
# Apply the function to assign EphMRA codes
antibodies['EphMRA Code'] = antibodies['Mechanism of Action'].apply(
    lambda x: assign_ephmra_code(x, ephmra_keywords)
)

# Show results
print(antibodies[['Mechanism of Action', 'EphMRA Code']].head(10))
print(f"\nEphMRA Code distribution:")
print(antibodies['EphMRA Code'].value_counts())

In [ ]:
def assign_ephmra_code(moa, ephmra_keywords):
    """
    Assign EphMRA code based on MOA keywords.
    
    Exceptions:
    - If MOA is 'Unknown' or 'Unclassified', pass through as-is.
    - If 2 or more codes among L1G1-L1G5 match, assign L1G9.
    - If no match found, assign V3 (All Other Therapeutic Products).
    """
    if pd.isna(moa) or moa == '-' or moa == '':
        return 'Unknown'
    
    moa_str = str(moa).strip()
    
    # Pass through Unknown and Unclassified as-is
    if moa_str.lower() in ['unknown', 'unclassified']:
        return moa_str
    
    moa_lower = moa_str.lower()
    matches_in_order = []

    # Capture matches while preserving dictionary order.
    for code, keywords in ephmra_keywords.items():
        for keyword in keywords:
            if keyword.lower() in moa_lower:
                matches_in_order.append(code)
                break

    if not matches_in_order:
        return 'V3 (All Other Therapeutic Products)'

    # Apply antineoplastic multi-match exception.
    l1g_target_prefixes = {'L1G1', 'L1G2', 'L1G3', 'L1G4', 'L1G5'}
    matched_l1g = {
        code.split(' ')[0]
        for code in matches_in_order
        if code.split(' ')[0] in l1g_target_prefixes
    }
    if len(matched_l1g) >= 2:
        for code in ephmra_keywords:
            if code.startswith('L1G9'):
                return code
        return 'L1G9 (Monoclonal Antibody Antineoplastics, Other)'

    return matches_in_order[0]

In [ ]:
# Load the antibody list
antibodies = pd.read_csv('List of Antibodies by MOA excel.csv')

print(f"Loaded {len(antibodies)} antibodies")
print(f"Columns: {list(antibodies.columns)}")
print(antibodies.head(3))

In [ ]:
# Define EphMRA code mapping with keywords
# Format: EphMRA code as key, list of keywords to search for (case-insensitive)
# Add your keywords here based on the EphMRA classification system

ephmra_keywords = {
    "G2x9 (Other gynaecologicals)": [
        "Neurokinin 3 (NK-3) receptor antagonist",
        "NK-3 receptor antagonist"
    ],
    "J6H7 (Coronavirus Immunoglobulin)": [
        "Coronavirus",

    "L1G1 (Monoclonal Antibody antineoplastics, CD20)": [
        "CD20",
        "rituximab",
        "obinutuzumab"
    ],
    "L1G2 (Monoclonal Antibody antineoplastics, VEGF/VEGFR)": [
        "VEGF",
        "VEGFR",
        "vascular endothelial growth factor",
        "vascular endothelial growth factor receptor",
        "bevacizumab",
        "ramucirumab"
    ],
    "L1G3 (Monoclonal Antibody antineoplastics, HER-2)": [
        "HER-2",
        "human epidermal growth factor receptor 2",
        "trastuzumab",
        "pertuzumab"
    ],
    "L1G4 (Monoclonal Antibody antineoplastics, EGFR)": [
        "EGFR",
        "epidermal growth factor receptor",
        "cetuximab",
        "panitumumab"
    ],
    "L1G5 (Monoclonal Antibody antineoplastics, PD-1/PD-L1)": [
        "PD-1",
        "PD-L1",
        "programmed cell death protein 1",
        "programmed death-ligand 1",
        "nivolumab",
        "pembrolizumab",
        "atezolizumab",
        "durvalumab"
    ],
    "L1G9 (Monoclonal Antibody antineoplastics, Other)": [
        "Natural Killer Cell",
        "NK cell",
        "Natural Killer",
        "Natural Killer Cell Stimulant",
        "NK Recruiter",
        "Natural killer (NK) cell stimulant",
        "T-cell engager",

    ],
    "L4B (Anti-TNF Products)": [
        "TNF",
        "TL1A",
        "TNF-alpha",
        "TNFa",
        "tumor necrosis factor",
        "tumour necrosis factor"
    ],
    "L4C (Interleukin Inhibitors)": [
        "interleukin",
        "IL-"
    ],
    "C10A4 (PCSK9 Inhibitors)": [
        "PCSK9",
        "proprotein convertase"
    ],
    "N2 (Analgesics)": [
        "Nav1.7",
        "NaV1.8",
        "Kv1.3",
    ],
    "N2C2 (Antimigraine CGRP Antagonists)": [
        "CGRP",
        "calcitonin gene-related peptide"
    ],
    "N7F (Drugs Used in Opioid Dependence)": [
        "Opioid receptor anatgonist"
    ],
    # Add more codes below
}